In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    to_date,
    trim,
    upper,
    when,
    round
)

spark = SparkSession.builder.getOrCreate()

# --------------------------------------------------
# Paths / Tables
# --------------------------------------------------

silver_checkpoint = "/Volumes/workspace/ibm/v10/checkpoints/silver_sales9"
silver_table = "workspace.default.capstone_silver_sales9"

bronze_table = "workspace.default.capstone_bronze_sales9"

# Remove stale checkpoint if needed
try:
    dbutils.fs.rm(silver_checkpoint, recurse=True)
except Exception:
    pass


# --------------------------------------------------
# Read Bronze as Stream
# --------------------------------------------------

bronze_df = (
    spark.readStream
         .table(bronze_table)
)


# --------------------------------------------------
# Silver Transformation
# --------------------------------------------------

silver_df = (
    bronze_df

    # Clean string columns
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", upper(trim(col("state"))))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
    .withColumn("payment_method", upper(trim(col("payment_method"))))
    .withColumn("order_status", upper(trim(col("order_status"))))

    # Convert data types
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("unit_price", col("unit_price").cast("double"))
    .withColumn("discount_pct", col("discount_pct").cast("double"))
    .withColumn("gross_amount", col("gross_amount").cast("double"))
    .withColumn("discount_amount", col("discount_amount").cast("double"))
    .withColumn("net_amount", col("net_amount").cast("double"))

    # Recalculate financial fields for consistency
    .withColumn(
        "gross_amount",
        round(col("quantity") * col("unit_price"), 2)
    )

    .withColumn(
        "discount_amount",
        round(
            col("gross_amount") * (col("discount_pct") / 100),
            2
        )
    )

    .withColumn(
        "net_amount",
        round(
            col("gross_amount") - col("discount_amount"),
            2
        )
    )

    # Keep valid records
    .filter(col("order_id").isNotNull())
    .filter(col("order_date").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("quantity") > 0)
    .filter(col("unit_price") >= 0)
)


# --------------------------------------------------
# Write Silver
# --------------------------------------------------

silver_query = (
    silver_df.writeStream
             .format("delta")
             .option("checkpointLocation", silver_checkpoint)
             .option("mergeSchema", "true")
             .outputMode("append")
             .trigger(availableNow=True)
             .toTable(silver_table)
)

silver_query.awaitTermination()

print(f"Silver table loaded successfully: {silver_table}")